# 2 - Guide d'imputation d'observations à haute fréquence pour des données à fréquences mixtes 

Ce notebook présente les fonctionalités de la classe `HighFrequencyImputer` du module `tsforecast.frequency`, qui permet l'imputation de valeurs haute fréquence à partir de séries à basse fréquence dans des jeux de données à fréquences mixtes.

## Table des Matières

- [1 - Importation des modules](#1---importation-des-modules)
- [2 - Création des jeux de données](#2---création-des-jeux-de-données)
    - [2.1 - Jeu de données de séries temporelles](#21---jeu-de-données-de-séries-temporelles)
    - [2.2 - Jeu de données de panel](#22---jeu-de-données-de-panel)
    - [2.3 - Visualisation des données brutes](#23---visualisation-des-données-brutes)
- [3 - Combinaisons de keep_lower_frequencies et cascade_refitting](#3---combinaisons-de-keep_lower_frequencies-et-cascade_refitting)
    - [3.1 - Comprendre les paramètres](#31---comprendre-les-paramètres)
    - [3.2 - Scénario A : keep_lower_frequencies=False, cascade_refitting=False](#32---scénario-a--keep_lower_frequenciesfalse-cascade_refittingfalse)
    - [3.3 - Scénario B : keep_lower_frequencies=True, cascade_refitting=False](#33---scénario-b--keep_lower_frequenciestrue-cascade_refittingfalse)
    - [3.4 - Scénario C : keep_lower_frequencies=False, cascade_refitting=True](#34---scénario-c--keep_lower_frequenciesfalse-cascade_refittingtrue)
    - [3.5 - Scénario D : keep_lower_frequencies=True, cascade_refitting=True](#35---scénario-d--keep_lower_frequenciestrue-cascade_refittingtrue)
    - [3.6 - Comparaison visuelle des scénarios](#36---comparaison-visuelle-des-scénarios)
- [4 - Fenêtres d'imputation et seuil d'attrition](#4---fenêtres-dimputation-et-seuil-dattrition)
    - [4.1 - Comprendre la fenêtre P1](#41---comprendre-la-fenêtre-p1)
    - [4.2 - Les quatre scopes d'imputation](#42---les-quatre-scopes-dimputation)
    - [4.3 - Impact du seuil d'attrition](#43---impact-du-seuil-dattrition)
    - [4.4 - Imputation directe vs imputation des valeurs manquantes d'abord](#44---imputation-directe-vs-imputation-des-valeurs-manquantes-dabord)
- [5 - Résumé et tableau récapitulatif](#5---résumé-et-tableau-récapitulatif)
    - [5.1 - Tableau récapitulatif des paramètres](#51---tableau-récapitulatif-des-paramètres)
    - [5.2 - Recommandations par cas d'usage](#52---recommandations-par-cas-dusage)
    - [5.3 - Points clés à retenir](#53---points-clés-à-retenir)

## 1 - Importation des modules <a id="1---importation-des-modules"></a>

Cette section présente les modules nécessaires pour utiliser le `HighFrequencyImputer`.

In [ ]:
# Importation des modules
# Modules de base
import warnings
from typing import Dict, List, Optional

# Manipulation de données
import numpy as np
import pandas as pd

# Graphiques
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

# Sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

# Module tsforecast
from tsforecast.frequency import HighFrequencyImputer
from tsforecast.frequency.provenance import ProvenanceType, ImputationProvenanceTracker
from tsforecast.frequency.imputation_window import P1WindowCalculator, ImputationScope

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Affichage
print("Modules importés avec succès !")

## 2 - Création des jeux de données <a id="2---création-des-jeux-de-données"></a>

Nous allons créer deux jeux de données fictifs :
1. Un jeu de **séries temporelles** simples
2. Un jeu de **données de panel** (plusieurs entités)

Chaque jeu contient des variables à différentes fréquences (mensuelle, trimestrielle, annuelle) avec des délais de publication variables simulant des situations réelles.

### 2.1 - Jeu de données de séries temporelles <a id="21---jeu-de-données-de-séries-temporelles"></a>

Ce jeu de données représente des indicateurs macroéconomiques typiques d'un pays :
- **PIB** : Publication trimestrielle avec délai de 2 mois
- **Inflation (IPC)** : Publication mensuelle avec délai de 1 mois
- **Taux de chômage** : Publication mensuelle avec délai de 1 mois
- **Production industrielle** : Publication mensuelle (disponible rapidement)
- **Balance commerciale annuelle** : Publication annuelle avec délai de 3 mois

In [ ]:
# Fonction de création de séries temporelles
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-06-30',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.
    
    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.
        
    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)
    
    # Création de l'index mensuel (fréquence cible)
    dates = pd.date_range(start=start_date, end=end_date, freq='ME')
    n_periods = len(dates)
    
    # Initialisation du DataFrame
    df = pd.DataFrame(index=dates)
    df.index.name = 'date'
    
    # ----- Variables mensuelles -----
    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise
    
    # Inflation mensuelle (IPC, entre 0.5% et 4%)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)
    
    # Taux de chômage (mensuel, entre 5% et 12%)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),  # Choc économique
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)
    
    # ----- Variable trimestrielle : PIB -----
    # Le PIB n'est disponible qu'aux fins de trimestre
    pib_base = 2500
    pib_growth_quarterly = 0.5  # Croissance trimestrielle moyenne
    df['pib_trimestriel'] = np.nan
    
    quarter_end_months = [3, 6, 9, 12]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_end_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1
    
    # ----- Variable annuelle : Balance commerciale -----
    df['balance_commerciale_annuelle'] = np.nan
    
    for i, date in enumerate(dates):
        if date.month == 12:
            year_factor = (date.year - 2018)
            base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
            df.loc[date, 'balance_commerciale_annuelle'] = base_balance
    
    # ----- Simulation des délais de publication -----
    # Délai de 1 mois pour l'inflation et le chômage
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan
    
    # Délai de 2 mois pour le PIB
    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan
    
    # Délai de 3 mois pour la balance commerciale annuelle
    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan
    
    # ----- Simulation de données historiques limitées -----
    # La production industrielle n'est disponible qu'à partir de 2019
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan
    
    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

# Affichage
print("=" * 80)
print("JEU DE DONNÉES DE SÉRIES TEMPORELLES")
print("=" * 80)
print(f"\nPériode : {df_timeseries.index.min().strftime('%Y-%m')} à {df_timeseries.index.max().strftime('%Y-%m')}")
print(f"Nombre d'observations : {len(df_timeseries)}")
print(f"Colonnes : {list(df_timeseries.columns)}")
print("\n--- Statistiques descriptives ---")
print(df_timeseries.describe().round(2))
print("\n--- Aperçu des dernières lignes ---")
display(df_timeseries.tail(15))

### 2.2 - Jeu de données de panel

<a id="22---jeu-de-données-de-panel"></a>

Ce jeu de données représente les mêmes indicateurs pour trois pays de la zone euro : France, Allemagne et Italie. Chaque pays a des caractéristiques légèrement différentes et des profondeurs historiques variables pour certaines séries.

In [ ]:
# Fonction de création d'un jeu de données de panel fictif
def create_panel_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-06-30',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.
    
    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.
        
    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)
    
    # Définition des pays et leurs caractéristiques
    countries = {
        'France': {
            'pib_base': 2800,
            'inflation_base': 1.5,
            'chomage_base': 8.0,
            'prod_ind_start': '2018-06-01'  # Historique complet
        },
        'Allemagne': {
            'pib_base': 3500,
            'inflation_base': 1.2,
            'chomage_base': 5.5,
            'prod_ind_start': '2019-01-01'  # Historique partiel
        },
        'Italie': {
            'pib_base': 2200,
            'inflation_base': 1.8,
            'chomage_base': 10.5,
            'prod_ind_start': '2019-06-01'  # Historique plus court
        }
    }
    
    # Initialisation des dates et du nombre de périodes
    dates = pd.date_range(start=start_date, end=end_date, freq='ME')
    n_periods = len(dates)
    
    # Initialisation de la liste des jeux de données pour l'ensemble des pays
    all_data = []
    
    # Parcours des pays
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)
        
        # Création du DataFrame pour ce pays
        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country
        
        # Production industrielle (mensuelle)
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        
        # Données non disponibles avant une certaine date
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan
        
        # Inflation (mensuelle)
        infl_trend = np.linspace(
            params['inflation_base'], 
            params['inflation_base'] + np.random.uniform(0.5, 2.0), 
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)
        
        # Taux de chômage (mensuel)
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)
        
        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [3, 6, 9, 12]
        quarter_idx = 0
        for i, date in enumerate(dates):
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1
        
        # Balance commerciale annuelle
        df_country['balance_commerciale_annuelle'] = np.nan
        for date in dates:
            if date.month == 12:
                year_factor = (date.year - 2018)
                base = -20 + np.random.uniform(-10, 10) + year_factor * 2
                df_country.loc[date, 'balance_commerciale_annuelle'] = base
        
        # Simulation des délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan
        
        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan
        
        all_data.append(df_country)
    
    # Concaténation et création du MultiIndex
    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()
    
    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

# Affichage
print("=" * 80)
print("JEU DE DONNÉES DE PANEL")
print("=" * 80)
print(f"\nEntités (pays) : {df_panel.index.get_level_values('country').unique().tolist()}")
print(f"Colonnes : {list(df_panel.columns)}")
print(f"Shape : {df_panel.shape}")
print("\n--- Statistiques par pays ---")
display(df_panel.groupby('country').agg(['count', 'mean']).round(2))
print("\n--- Aperçu pour la France ---")
display(df_panel.loc['France'].tail(10))

### 2.3 - Visualisation des données brutes

<a id="23---visualisation-des-données-brutes"></a>

Visualisons les données pour comprendre leur structure et la répartition des valeurs manquantes.

In [ ]:
# Fonction de visualitaion des données
def plot_data_availability(df: pd.DataFrame, title: str = "Disponibilité des données"):
    """Plot data availability heatmap showing NaN patterns.
    
    Args:
        df: DataFrame to visualize.
        title: Plot title.
    """
    # Gestion du MultiIndex pour panel
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    # Création de la matrice de disponibilité
    availability = plot_df.notna().astype(int)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Création du heatmap avec couleurs personnalisées
    cmap = ListedColormap(['#ffcccc', '#90EE90'])  # Rouge clair pour NaN, vert pour disponible
    
    im = ax.imshow(availability.T, aspect='auto', cmap=cmap, interpolation='nearest')
    
    # Configuration des axes
    ax.set_yticks(range(len(plot_df.columns)))
    ax.set_yticklabels(plot_df.columns)
    
    # Affichage d'un sous-ensemble des dates
    n_ticks = 12
    tick_indices = np.linspace(0, len(plot_df) - 1, n_ticks, dtype=int)
    ax.set_xticks(tick_indices)
    ax.set_xticklabels([plot_df.index[i].strftime('%Y-%m') for i in tick_indices], rotation=45, ha='right')
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Variable')
    ax.set_title(title)
    
    # Légende
    legend_elements = [
        mpatches.Patch(facecolor='#90EE90', label='Données disponibles'),
        mpatches.Patch(facecolor='#ffcccc', label='Valeurs manquantes (NaN)')
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1))
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques de disponibilité
    print("\nStatistiques de disponibilité :")
    availability_pct = (plot_df.notna().sum() / len(plot_df) * 100).round(1)
    for col, pct in availability_pct.items():
        print(f"  {col}: {pct}% disponible")


# Visualisation pour les séries temporelles
print("\n" + "=" * 80)
print("VISUALISATION DU JEU DE DONNÉES DE SÉRIES TEMPORELLES")
print("=" * 80)
plot_data_availability(df_timeseries, "Disponibilité des données - Séries temporelles")

# Visualisation pour le panel (France seulement pour simplifier)
print("\n" + "=" * 80)
print("VISUALISATION DU JEU DE DONNÉES DE PANEL (France)")
print("=" * 80)
plot_data_availability(df_panel.loc['France'], "Disponibilité des données - France (Panel)")

In [ ]:
# Fonction d'affichage des séries par fréquence
def plot_series_with_frequencies(df: pd.DataFrame, title: str = "Variables par fréquence"):
    """Plot time series colored by their frequency.
    
    Args:
        df: DataFrame to visualize.
        title: Plot title.
    """
    # Gestion du MultiIndex
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    # Détection automatique des fréquences
    freq_colors = {
        'Mensuelle': '#2ecc71',
        'Trimestrielle': '#3498db',
        'Annuelle': '#e74c3c'
    }
    
    fig, axes = plt.subplots(len(plot_df.columns), 1, figsize=(14, 3 * len(plot_df.columns)), sharex=True)
    if len(plot_df.columns) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, plot_df.columns):
        series = plot_df[col].dropna()
        
        # Estimation de la fréquence
        if len(series) > 1:
            avg_gap = (series.index[1:] - series.index[:-1]).mean().days
            if avg_gap < 45:
                freq_label = 'Mensuelle'
            elif avg_gap < 120:
                freq_label = 'Trimestrielle'
            else:
                freq_label = 'Annuelle'
        else:
            freq_label = 'Mensuelle'
        
        color = freq_colors[freq_label]
        
        # Tracé
        ax.plot(series.index, series.values, color=color, linewidth=1.5, marker='o', markersize=3)
        ax.fill_between(series.index, series.values, alpha=0.1, color=color)
        ax.set_ylabel(col, fontsize=9)
        ax.legend([f'{freq_label}'], loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    axes[0].set_title(title, fontsize=12, fontweight='bold')
    axes[-1].set_xlabel('Date')
    
    plt.tight_layout()
    plt.show()


# Visualisation des séries par fréquence
plot_series_with_frequencies(df_timeseries, "Variables macroéconomiques par fréquence")

---

## 3 - Combinaisons de keep_lower_frequencies et cascade_refitting

<a id="3---combinaisons-de-keep_lower_frequencies-et-cascade_refitting"></a>

Cette section illustre l'impact des paramètres `keep_lower_frequencies` et `cascade_refitting` sur le processus d'imputation.

### 3.1 - Comprendre les paramètres <a id="31---comprendre-les-paramètres"></a>

#### `keep_lower_frequencies` (bool)

Ce paramètre contrôle la **structure de sortie** :

| Valeur | Comportement | Structure de sortie |
|--------|--------------|--------------------|
| `False` | Supprime les fréquences inférieures à la cible | Index simple `(Date)` ou `(Entity, Date)` |
| `True` | Conserve toutes les fréquences intermédiaires | MultiIndex `(Frequency, Date)` ou `(Entity, Frequency, Date)` |

#### `cascade_refitting` (bool)

Ce paramètre contrôle la **stratégie d'entraînement** :

| Valeur | Comportement | Impact |
|--------|--------------|--------|
| `False` | Un seul entraînement, prédiction directe vers la fréquence cible | Plus rapide |
| `True` | Réentraînement après chaque niveau de fréquence | Plus lent, mais utilise l'information des imputations précédentes |

#### Schéma conceptuel de l'imputation en cascade

```
┌───────────────────────────────────────────────────────────────────────────────┐
│                    IMPUTATION EN CASCADE                                      │
├───────────────────────────────────────────────────────────────────────────────┤
│                                                                               │
│  DONNÉES BRUTES                                                               │
│  ├── Annuelles (A)      ●───────────────●                                     │
│  ├── Trimestrielles (Q) ●───●───●───●───●───●                                 │
│  └── Mensuelles (M)     ●●●●●●●●●●●●●●●●●●●●●●●●                              │
│                                                                               │
│  ══════════════════════════════════════════════════════════════════════════   │
│                                                                               │
│  ÉTAPE 1: Imputation A → Q                                                    │
│  ├── Agrégation des M et Q à la fréquence A                                   │
│  ├── Entraînement du modèle sur les vraies valeurs A                          │
│  └── Prédiction des valeurs A à la fréquence Q                                │
│                                                                               │
│  [Si cascade_refitting=True : Réentraînement sur les valeurs A + Q imputées]     │
│                                                                               │
│  ÉTAPE 2: Imputation Q → M                                                    │
│  ├── Agrégation des M à la fréquence Q                                        │
│  ├── Entraînement sur vraies Q (+ imputées si cascade_refitting=True)            │
│  └── Prédiction des valeurs Q à la fréquence M                                │
│                                                                               │
│  ══════════════════════════════════════════════════════════════════════════   │
│                                                                               │
│  SORTIE (selon keep_lower_frequencies) :                                      │
│  ├── False → Uniquement les données à fréquence M                             │
│  └── True  → Toutes les fréquences (A, Q, M) en MultiIndex                    │
│                                                                               │
└───────────────────────────────────────────────────────────────────────────────┘
```

### 3.2 - Configuraion avec keep_lower_frequencies=False, cascade_refitting=False <a id="32---scénario-a--keep_lower_frequenciesfalse-cascade_refittingfalse"></a>

**Configuration la plus simple et la plus rapide.** 

- Prédiction directe de la fréquence source vers la fréquence cible
- Pas de réentraînement intermédiaire
- Sortie uniquement à la fréquence cible


In [ ]:
# Schéma
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis('off')

# Entrée
ax.add_patch(plt.Rectangle((0.5, 3), 2, 0.8, facecolor='#3498db', alpha=0.7))
ax.text(1.5, 3.4, 'Données\nmixtes', ha='center', va='center', fontsize=10, fontweight='bold')

# Flèche
ax.annotate('', xy=(4, 3.4), xytext=(2.7, 3.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(3.35, 3.7, 'cascade_refitting=False\n(1 seul entraînement)', ha='center', fontsize=8)

# Imputation directe
ax.add_patch(plt.Rectangle((4.2, 3), 2.5, 0.8, facecolor='#e74c3c', alpha=0.7))
ax.text(5.45, 3.4, 'Imputation\ndirecte A,Q→M', ha='center', va='center', fontsize=10, fontweight='bold')

# Flèche
ax.annotate('', xy=(8, 3.4), xytext=(6.9, 3.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(7.45, 3.7, 'keep_lower_frequencies=False\n(supprime A et Q)', ha='center', fontsize=8)

# Sortie
ax.add_patch(plt.Rectangle((8.2, 3), 1.5, 0.8, facecolor='#2ecc71', alpha=0.7))
ax.text(8.95, 3.4, 'Sortie\n(M seul)', ha='center', va='center', fontsize=10, fontweight='bold')

# Titre
ax.text(5, 4.2, 'Scénario A : Imputation directe, sortie simple', ha='center', fontsize=14, fontweight='bold')

# Provenance
ax.add_patch(plt.Rectangle((1, 0.5), 8, 1.8, facecolor='#f8f9fa', edgecolor='#dee2e6', linewidth=2))
ax.text(5, 2.0, 'Provenance des valeurs imputées :', ha='center', fontsize=11, fontweight='bold')
ax.text(5, 1.5, '• Variables mensuelles : ORIGINAL (inchangées)', ha='center', fontsize=9)
ax.text(5, 1.1, '• pib_trimestriel → mensuel : MODEL_ON_TRUE (modèle entraîné sur vraies valeurs)', ha='center', fontsize=9)
ax.text(5, 0.7, '• balance_commerciale_annuelle → mensuel : MODEL_ON_TRUE', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

1. Détection des fréquences :
   - production_industrielle : Mensuelle (M) → pas d'imputation
   - inflation_ipc : Mensuelle (M) → pas d'imputation
   - taux_chomage : Mensuelle (M) → pas d'imputation
   - pib_trimestriel : Trimestrielle (Q) → imputation Q→M
   - balance_commerciale_annuelle : Annuelle (A) → imputation A→M

2. Processus d'imputation :
   - Entraînement unique sur P1 avec vraies valeurs
   - Prédiction directe : A→M et Q→M (en parallèle)
   
3. Sortie :
   - Index DatetimeIndex simple
   - Toutes les variables à fréquence mensuelle
   - Les fréquences intermédiaires ne sont PAS conservées

In [ ]:
# Configuration simple avec keep_lower_frequencies=False, cascade_refitting=False

# Initialisation de l'imputer
imputer_a = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=False,
    imputation_scope='strict'
)

# Application sur les séries temporelles
df_imputed_a = imputer_a.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape avant imputation : {df_timeseries.shape}")
print(f"Shape après imputation : {df_imputed_a.shape}")

df_imputed_a.head()

### 3.3 - Configuration avec keep_lower_frequencies=True, cascade_refitting=False <a id="33---scénario-b--keep_lower_frequenciestrue-cascade_refittingfalse"></a>

**Conservation de toutes les fréquences intermédiaires.**

- Prédiction directe vers la fréquence cible
- Sortie avec MultiIndex incluant le niveau de fréquence

In [ ]:
# Configuration avec keep_lower_frequencies=True, cascade_refitting=False
# Initialisation de l'imputer
imputer_b = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=True,
    cascade_refitting=False,
    imputation_scope='strict'
)
# Imputation
df_imputed_b = imputer_b.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape après imputation : {df_imputed_b.shape}")
print(f"Type d'index : {type(df_imputed_b.index).__name__}")

df_imputed_b.head()

### 3.4 - Configuration avec keep_lower_frequencies=False, cascade_refitting=True <a id="34---scénario-c--keep_lower_frequenciesfalse-cascade_refittingtrue"></a>

**Imputation en cascade avec réentraînement.**

- Les modèles sont réentraînés après chaque niveau de fréquence, de la plus faible à la plus élevées
- Les valeurs imputées aux fréquences inférieures sont utilisées pour améliorer les prédictions suivantes
- Sortie simple à la fréquence cible uniquement


In [ ]:
# Visualisation du flux
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# Titre
ax.text(7, 7.5, 'Scénario C : Cascade avec réentraînement, sortie simple', 
        ha='center', fontsize=14, fontweight='bold')

# Données initiales
y_start = 6.5
ax.add_patch(plt.Rectangle((0.5, y_start), 2.5, 0.8, facecolor='#3498db', alpha=0.7, edgecolor='black'))
ax.text(1.75, y_start + 0.4, 'Données mixtes\nA + Q + M', ha='center', va='center', fontsize=9, fontweight='bold')

# Flèche vers étape 1
ax.annotate('', xy=(4.5, y_start + 0.4), xytext=(3.2, y_start + 0.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Étape 1 : A → Q
y_step1 = 6.5
ax.add_patch(plt.Rectangle((4.5, y_step1), 4, 0.8, facecolor='#e74c3c', alpha=0.7, edgecolor='black'))
ax.text(6.5, y_step1 + 0.4, 'Étape 1 : Imputation A → Q\nEntraînement sur vraies A', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Flèche de réentraînement
ax.annotate('', xy=(8.5, y_step1 - 0.3), xytext=(8.5, y_step1),
            arrowprops=dict(arrowstyle='->', color='#9b59b6', lw=2))
ax.text(10.5, y_step1 - 0.5, '★ Réentraînement avec\nnouvelles valeurs Q', 
        ha='center', fontsize=8, color='#9b59b6', fontweight='bold')

# Flèche vers étape 2
ax.annotate('', xy=(6.5, y_step1 - 1.2), xytext=(6.5, y_step1 - 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Étape 2 : Q → M
y_step2 = 4.3
ax.add_patch(plt.Rectangle((4.5, y_step2), 4, 0.8, facecolor='#f39c12', alpha=0.7, edgecolor='black'))
ax.text(6.5, y_step2 + 0.4, 'Étape 2 : Imputation Q → M\nEntraînement sur vraies+imputées Q', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Flèche vers sortie
ax.annotate('', xy=(6.5, y_step2 - 1.2), xytext=(6.5, y_step2 - 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Sortie
y_out = 2.1
ax.add_patch(plt.Rectangle((4.5, y_out), 4, 0.8, facecolor='#2ecc71', alpha=0.7, edgecolor='black'))
ax.text(6.5, y_out + 0.4, 'Sortie : Données mensuelles (M)\nIndex DatetimeIndex simple', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Légende provenance
ax.add_patch(plt.Rectangle((0.5, 0.3), 13, 1.5, facecolor='#f8f9fa', edgecolor='#dee2e6', linewidth=2))
ax.text(7, 1.5, 'Provenance des valeurs :', ha='center', fontsize=10, fontweight='bold')
ax.text(4, 1.0, '• Étape 1 (A→Q) : MODEL_ON_TRUE', ha='left', fontsize=9, color='#e74c3c')
ax.text(4, 0.6, '• Étape 2 (Q→M) : MODEL_ON_MIXED', ha='left', fontsize=9, color='#f39c12')
ax.text(9, 1.0, '• Variables M originales : ORIGINAL', ha='left', fontsize=9, color='#2ecc71')

plt.tight_layout()
plt.show()

┌─────────────────────────────────────────────────────────────────────────────┐
│  ÉTAPE 1 : Traitement des variables annuelles (A)                           │
├─────────────────────────────────────────────────────────────────────────────┤
│  1.1 Agrégation des features mensuelles et trimestrielles à A               │
│  1.2 Entraînement du modèle sur P1 (vraies valeurs A uniquement)            │
│  1.3 Prédiction A → Q (fréquence intermédiaire)                             │
│  1.4 Marquage provenance : MODEL_ON_TRUE                                    │
└─────────────────────────────────────────────────────────────────────────────┘
                                      ↓
┌─────────────────────────────────────────────────────────────────────────────┐
│  ÉTAPE 2 : Traitement des variables trimestrielles (Q)                      │
├─────────────────────────────────────────────────────────────────────────────┤
│  2.1 Agrégation des features mensuelles à Q                                 │
│  2.2 Entraînement incluant les valeurs A et celles Q imputées à l'étape 1   │
│  2.3 Prédiction Q → M (fréquence cible)                                     │
│  2.4 Marquage provenance : MODEL_ON_MIXED (car entrainé sur imputées)       │
└─────────────────────────────────────────────────────────────────────────────┘
                                      ↓
┌─────────────────────────────────────────────────────────────────────────────┐
│  SORTIE : Données à fréquence mensuelle uniquement                          │
│  (les fréquences A et Q intermédiaires sont supprimées)                     │
└─────────────────────────────────────────────────────────────────────────────┘

In [ ]:
# Configuration avec keep_lower_frequencies=False, cascade_refitting=True

# Initialisation de l'imputer
imputer_c = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='strict'
)

# Imputation
df_imputed_c = imputer_c.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape après imputation : {df_imputed_b.shape}")
print(f"Type d'index : {type(df_imputed_b.index).__name__}")

df_imputed_c.head()

### 3.5 - Configuration avec keep_lower_frequencies=True, cascade_refitting=True <a id="35---scénario-d--keep_lower_frequenciestrue-cascade_refittingtrue"></a>

**Configuration la plus complète**

- Imputation en cascade avec réentraînement à chaque niveau
- Conservation de toutes les fréquences intermédiaires
- Maximum d'information disponible pour l'analyse


In [ ]:
# Configuration avec keep_lower_frequencies=True, cascade_refitting=True
# Initialisation de l'imputer
imputer_d = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=True,
    cascade_refitting=True,
    imputation_scope='extended_both',
    attrition_threshold=0.5
)
# Imputation
df_imputed_d = imputer_d.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape après imputation : {df_imputed_d.shape}")
print(f"Niveaux de l'index : {df_imputed_d.index.names}")
print(f"Fréquences disponibles : {df_imputed_d.index.get_level_values('frequency').unique().tolist()}")

# Accès à la matrice de provenance
print("\n--- Matrice de provenance ---")
display(imputer_d.imputation_provenance_.value_counts())

### 3.6 - Comparaison visuelle des scénarios

<a id="36---comparaison-visuelle-des-scénarios"></a>

Tableau comparatif des quatre configurations :

In [ ]:
# Tableau comparatif des scénarios
comparison_data = {
    'Scénario': ['A', 'B', 'C', 'D'],
    'keep_lower_frequencies': [False, True, False, True],
    'cascade_refitting': [False, False, True, True],
    'Structure sortie': [
        'Index simple (Date)',
        'MultiIndex (Freq, Date)',
        'Index simple (Date)',
        'MultiIndex (Freq, Date)'
    ],
    'Entraînements': ['1 seul', '1 seul', 'N (cascade)', 'N (cascade)'],
    'Vitesse': ['★★★★★', '★★★★☆', '★★★☆☆', '★★☆☆☆'],
    'Précision': ['★★☆☆☆', '★★☆☆☆', '★★★★☆', '★★★★★'],
    'Traçabilité': ['★★☆☆☆', '★★★★☆', '★★★☆☆', '★★★★★'],
    'Cas d\'usage': [
        'Prototypage rapide',
        'Analyse multi-échelle',
        'Production, précision',
        'Recherche complète'
    ]
}

df_comparison = pd.DataFrame(comparison_data)

print("=" * 100)
print("COMPARAISON DES SCÉNARIOS")
print("=" * 100)
display(df_comparison.set_index('Scénario'))

# Visualisation graphique
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Trade-off Vitesse vs Précision
ax1 = axes[0]
scenarios = ['A', 'B', 'C', 'D']
speed_scores = [5, 4, 3, 2]
precision_scores = [2, 2, 4, 5]
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

for s, sp, pr, c in zip(scenarios, speed_scores, precision_scores, colors):
    ax1.scatter(sp, pr, s=500, c=c, alpha=0.7, edgecolors='black', linewidth=2)
    ax1.annotate(f'Scénario {s}', (sp, pr), textcoords="offset points", 
                 xytext=(10, 10), fontsize=11, fontweight='bold')

ax1.set_xlabel('Vitesse (1=lent, 5=rapide)', fontsize=12)
ax1.set_ylabel('Précision (1=basse, 5=haute)', fontsize=12)
ax1.set_title('Trade-off Vitesse vs Précision', fontsize=13, fontweight='bold')
ax1.set_xlim(1, 6)
ax1.set_ylim(1, 6)
ax1.grid(True, alpha=0.3)

# Graphique 2 : Complexité de la structure de sortie
ax2 = axes[1]
bar_positions = [0, 1, 2, 3]
structure_complexity = [1, 2, 1, 2]  # 1=simple, 2=MultiIndex
retraining = [0, 0, 1, 1]  # 0=non, 1=oui

bar_width = 0.35
bars1 = ax2.bar([p - bar_width/2 for p in bar_positions], structure_complexity, 
                bar_width, label='Complexité index', color='#3498db', alpha=0.7)
bars2 = ax2.bar([p + bar_width/2 for p in bar_positions], retraining, 
                bar_width, label='Réentraînement', color='#e74c3c', alpha=0.7)

ax2.set_xticks(bar_positions)
ax2.set_xticklabels([f'Scénario {s}' for s in scenarios])
ax2.set_ylabel('Niveau de complexité', fontsize=12)
ax2.set_title('Caractéristiques par scénario', fontsize=13, fontweight='bold')
ax2.legend()
ax2.set_ylim(0, 2.5)

plt.tight_layout()
plt.show()

---

## 4 - Fenêtres d'imputation et seuil d'attrition

<a id="4---fenêtres-dimputation-et-seuil-dattrition"></a>

Cette section explore l'impact des paramètres `imputation_scope`, `attrition_threshold` et `train_on_partial_coverage` sur le processus d'imputation.

### 4.1 - Comprendre la fenêtre stricte

<a id="41---comprendre-la-fenêtre-p1"></a>

La **fenêtre stricte d'imputation** est la période temporelle où **toutes les séries ont des valeurs réelles** (non-NaN). C'est la fenêtre de référence pour l'entraînement des modèles d'imputation.

```
Temps →          2018    2019    2020    2021    2022    2023    2024
                   |       |       |       |       |       |       |
Variable A     ────●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●○○○────
Variable B     ────○○○○●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●────
Variable C     ────○○○○○○○○●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●○○○○────
Variable D     ────●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●○○○○○○○────
                           ↑──────────────────────────↑
                           |     fenêtre stricte      |
                  Début fenêtre stricte         Fin fenêtre stricte

● = Valeur disponible
○ = Valeur manquante (NaN)
```

In [ ]:
def visualize_strict_window(df: pd.DataFrame):
    """Visualize strict window calculation on the dataset.
    
    Args:
        df: DataFrame to analyze.
    """
    # Gestion du MultiIndex
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3, 1])
    
    # Graphique 1 : Disponibilité par variable
    ax1 = axes[0]
    
    n_cols = len(plot_df.columns)
    colors = plt.cm.tab10(np.linspace(0, 1, n_cols))
    
    for i, col in enumerate(plot_df.columns):
        # Création du masque de disponibilité
        mask = plot_df[col].notna()
        y_positions = np.where(mask, i + 0.4, np.nan)
        ax1.scatter(plot_df.index, y_positions, c=[colors[i]], s=3, alpha=0.8, label=col)
    
    # Calcul de P1 (approximatif pour illustration)
    all_available = plot_df.notna().all(axis=1)
    if all_available.any():
        p1_start = plot_df.index[all_available].min()
        p1_end = plot_df.index[all_available].max()
        
        # Zone P1
        ax1.axvspan(p1_start, p1_end, alpha=0.2, color='green', label='Fenêtre P1')
        ax1.axvline(p1_start, color='green', linestyle='--', linewidth=2, alpha=0.7)
        ax1.axvline(p1_end, color='green', linestyle='--', linewidth=2, alpha=0.7)
    
    ax1.set_yticks(range(n_cols))
    ax1.set_yticklabels(plot_df.columns)
    ax1.set_ylabel('Variables')
    ax1.set_title('Disponibilité des données et fenêtre P1', fontsize=13, fontweight='bold')
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
    
    # Graphique 2 : Taux d'attrition par date
    ax2 = axes[1]
    
    # Calcul du nombre de colonnes disponibles par date
    cols_available = plot_df.notna().sum(axis=1)
    attrition_rate = cols_available / len(plot_df.columns)
    
    ax2.fill_between(plot_df.index, attrition_rate, alpha=0.5, color='#3498db')
    ax2.plot(plot_df.index, attrition_rate, color='#2980b9', linewidth=1.5)
    
    # Lignes de seuil
    ax2.axhline(0.5, color='orange', linestyle='--', label='Seuil 50%')
    ax2.axhline(0.7, color='red', linestyle='--', label='Seuil 70%')
    ax2.axhline(1.0, color='green', linestyle='--', label='100% (P1)')
    
    ax2.set_ylabel('Taux de\ndisponibilité')
    ax2.set_xlabel('Date')
    ax2.set_ylim(0, 1.1)
    ax2.legend(loc='lower right', fontsize=8)
    ax2.set_title('Taux de couverture des données par date', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques
    if all_available.any():
        print(f"\n--- Statistiques de la fenêtre P1 ---")
        print(f"Début P1 : {p1_start.strftime('%Y-%m')}")
        print(f"Fin P1 : {p1_end.strftime('%Y-%m')}")
        print(f"Durée P1 : {all_available.sum()} observations")
    else:
        print("\n⚠️ Aucune période avec 100% de données disponibles !")


# Visualisation de la fenêtre stricte
print("=" * 80)
print("VISUALISATION DE LA FENÊTRE STRICTE")
print("=" * 80)
visualize_strict_window(df_timeseries)

### 4.2 - Les quatre scopes d'imputation

<a id="42---les-quatre-scopes-dimputation"></a>

Le paramètre `imputation_scope` contrôle comment la fenêtre P1 est étendue pour l'entraînement des modèles :

| Scope | Description | Fenêtre d'entraînement |
|-------|-------------|------------------------|
| `strict` | Utilise uniquement la fenêtre stricte | P1 |
| `extended_backward` | Étend la fenêtre stricte vers le passé | [Début étendu, Fin fenêtre stricte] |
| `extended_forward` | Étend la fenêtre stricte vers le futur | [Début fenêtre stricte, Fin étendue] |
| `extended_both` | Étend dans les deux directions | [Début étendu, Fin étendue] |

In [ ]:
def visualize_imputation_scopes():
    """Create a visual diagram of the four imputation scopes."""
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    
    scopes = [
        ('strict', 'Uniquement la fenêtre P1'),
        ('extended_backward', 'P1 + Extension vers le passé'),
        ('extended_forward', 'P1 + Extension vers le futur'),
        ('extended_both', 'P1 + Extension dans les deux sens')
    ]
    
    for ax, (scope, title) in zip(axes, scopes):
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 1)
        ax.axis('off')
        
        # Timeline de base
        ax.axhline(0.5, color='black', linewidth=2, xmin=0.05, xmax=0.95)
        
        # Marqueurs temporels
        for x, label in [(1, '2018'), (2.5, '2019'), (4, 'P1\nstart'), (6, 'P1\nend'), (7.5, '2023'), (9, '2024')]:
            ax.plot(x, 0.5, 'ko', markersize=6)
            ax.text(x, 0.3, label, ha='center', fontsize=8)
        
        # Zone P1 (toujours présente)
        ax.add_patch(plt.Rectangle((4, 0.45), 2, 0.1, facecolor='#2ecc71', alpha=0.8))
        ax.text(5, 0.7, 'P1', ha='center', fontsize=10, fontweight='bold', color='#27ae60')
        
        # Extensions selon le scope
        if scope == 'extended_backward' or scope == 'extended_both':
            # Extension vers le passé
            ax.add_patch(plt.Rectangle((2, 0.45), 2, 0.1, facecolor='#f39c12', alpha=0.6))
            ax.annotate('', xy=(2, 0.5), xytext=(4, 0.5),
                       arrowprops=dict(arrowstyle='<->', color='#e67e22', lw=2))
            ax.text(3, 0.75, 'Extension\n(avant)', ha='center', fontsize=8, color='#d35400')
        
        if scope == 'extended_forward' or scope == 'extended_both':
            # Extension vers le futur
            ax.add_patch(plt.Rectangle((6, 0.45), 1.5, 0.1, facecolor='#e74c3c', alpha=0.6))
            ax.annotate('', xy=(6, 0.5), xytext=(7.5, 0.5),
                       arrowprops=dict(arrowstyle='<->', color='#c0392b', lw=2))
            ax.text(6.75, 0.75, 'Extension\n(après)', ha='center', fontsize=8, color='#a93226')
        
        # Titre
        ax.text(0.5, 0.9, f"{scope}", fontsize=11, fontweight='bold', transform=ax.transAxes)
        ax.text(0.5, 0.05, title, fontsize=9, ha='center', transform=ax.transAxes, style='italic')
    
    plt.suptitle('Les quatre scopes d\'imputation', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


print("=" * 80)
print("LES QUATRE FENETRES D'IMPUTATION")
print("=" * 80)
visualize_imputation_scopes()

### 4.3 - Impact du seuil d'attrition

<a id="43---impact-du-seuil-dattrition"></a>

Le paramètre `attrition_threshold` (entre 0 et 1) définit le pourcentage minimum de colonnes qui doivent avoir des données pour qu'une date soit incluse dans la fenêtre d'entraînement étendue.

**Exemples :**
- `attrition_threshold=0.5` : Au moins 50% des colonnes doivent avoir des données
- `attrition_threshold=0.8` : Au moins 80% des colonnes doivent avoir des données
- `attrition_threshold=1.0` : Équivalent à `strict` (100% requis)

In [ ]:
def demonstrate_attrition_impact(df: pd.DataFrame):
    """Demonstrate the impact of different attrition thresholds.
    
    Args:
        df: DataFrame to analyze.
    """
    # Gestion du MultiIndex
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    # Calcul du taux de couverture par date
    coverage_rate = plot_df.notna().sum(axis=1) / len(plot_df.columns)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    thresholds = [0.3, 0.5, 0.7, 0.9]
    
    for ax, threshold in zip(axes.flat, thresholds):
        # Identification des périodes valides
        valid_mask = coverage_rate >= threshold
        
        # Tracé de la couverture
        ax.fill_between(plot_df.index, coverage_rate, alpha=0.3, color='#3498db')
        ax.plot(plot_df.index, coverage_rate, color='#2980b9', linewidth=1.5, label='Couverture')
        
        # Mise en évidence des périodes valides
        ax.fill_between(plot_df.index, 0, 1, where=valid_mask, 
                       alpha=0.2, color='green', label='Fenêtre valide')
        
        # Seuil
        ax.axhline(threshold, color='red', linestyle='--', linewidth=2, 
                  label=f'Seuil = {threshold:.0%}')
        
        # Statistiques
        n_valid = valid_mask.sum()
        pct_valid = n_valid / len(plot_df) * 100
        
        ax.set_title(f'attrition_threshold = {threshold}\n({n_valid} obs. valides, {pct_valid:.1f}%)', 
                    fontsize=11, fontweight='bold')
        ax.set_ylabel('Taux de couverture')
        ax.set_ylim(0, 1.1)
        ax.legend(loc='lower right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Impact du seuil d\'attrition sur la fenêtre d\'entraînement', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Tableau récapitulatif
    print("\n--- Récapitulatif par seuil ---")
    summary_data = []
    for threshold in [0.3, 0.5, 0.7, 0.9, 1.0]:
        valid = coverage_rate >= threshold
        if valid.any():
            start = plot_df.index[valid].min().strftime('%Y-%m')
            end = plot_df.index[valid].max().strftime('%Y-%m')
        else:
            start = end = 'N/A'
        summary_data.append({
            'Seuil': f'{threshold:.0%}',
            'Observations valides': valid.sum(),
            'Début': start,
            'Fin': end
        })
    
    display(pd.DataFrame(summary_data))


print("=" * 80)
print("IMPACT DU SEUIL D'ATTRITION")
print("=" * 80)
demonstrate_attrition_impact(df_timeseries)

### 4.4 - Imputation directe vs imputation des valeurs manquantes d'abord

<a id="44---imputation-directe-vs-imputation-des-valeurs-manquantes-dabord"></a>

Le paramètre `train_on_partial_coverage` contrôle si les valeurs imputées sont utilisées pour l'entraînement des modèles **en dehors de la fenêtre P1**.

| `train_on_partial_coverage` | Comportement |
|---------------------------|---------------|
| `False` | Utilise uniquement les vraies valeurs pour l'entraînement |
| `True` | Utilise vraies valeurs + valeurs imputées précédemment |


╔═══════════════════════════════════════════════════════════════════════════════╗
║  COMPARAISON DES STRATÉGIES D'ENTRAÎNEMENT                                    ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  train_on_partial_coverage = False                                            ║
║  ─────────────────────────────────                                            ║
║  ✓ Plus conservateur et prudent                                               ║
║  ✓ Pas de propagation d'erreurs d'imputation                                  ║
║  ✗ Moins de données d'entraînement                                            ║
║  ✗ Peut être insuffisant si P1 est courte                                     ║
║                                                                               ║
║  → Recommandé quand : P1 est suffisamment longue, données de haute qualité    ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  train_on_partial_coverage = True                                             ║
║  ────────────────────────────────                                             ║
║  ✓ Plus de données d'entraînement                                             ║
║  ✓ Meilleure couverture des patterns temporels                                ║
║  ✗ Risque de propagation d'erreurs                                            ║
║  ✗ Provenance mixte des valeurs (MODEL_ON_MIXED)                              ║
║                                                                               ║
║  → Recommandé quand : P1 est courte, besoin de plus de données                ║
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
def visualize_training_strategies():
    """Visualize the two training strategies."""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Stratégie 1 : train_on_partial_coverage=False
    ax1 = axes[0]
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 6)
    ax1.axis('off')
    ax1.set_title('train_on_partial_coverage = False\n(Entraînement sur vraies valeurs uniquement)', 
                 fontsize=11, fontweight='bold')
    
    # Données d'entrée
    ax1.add_patch(plt.Rectangle((0.5, 4.5), 9, 1), fc='#ecf0f1', ec='black')
    ax1.text(5, 5, 'Données avec NaN', ha='center', va='center', fontsize=10)
    
    # Flèche
    ax1.annotate('', xy=(5, 3.8), xytext=(5, 4.3),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Séparation P1 / hors P1
    ax1.add_patch(plt.Rectangle((0.5, 2.5), 4, 1), fc='#2ecc71', alpha=0.6, ec='black')
    ax1.text(2.5, 3, 'P1\n(vraies valeurs)', ha='center', va='center', fontsize=9)
    
    ax1.add_patch(plt.Rectangle((4.5, 2.5), 5, 1), fc='#e74c3c', alpha=0.3, ec='black')
    ax1.text(7, 3, 'Hors P1\n(non utilisé)', ha='center', va='center', fontsize=9, color='gray')
    
    # Flèche
    ax1.annotate('', xy=(5, 1.8), xytext=(5, 2.3),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Entraînement
    ax1.add_patch(plt.Rectangle((2, 0.5), 6, 1), fc='#3498db', alpha=0.7, ec='black')
    ax1.text(5, 1, 'Modèle entraîné sur P1 uniquement', ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Stratégie 2 : train_on_partial_coverage=True
    ax2 = axes[1]
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 6)
    ax2.axis('off')
    ax2.set_title('train_on_partial_coverage = True\n(Entraînement sur vraies + imputées)', 
                 fontsize=11, fontweight='bold')
    
    # Données d'entrée
    ax2.add_patch(plt.Rectangle((0.5, 4.5), 9, 1), fc='#ecf0f1', ec='black')
    ax2.text(5, 5, 'Données avec NaN', ha='center', va='center', fontsize=10)
    
    # Flèche
    ax2.annotate('', xy=(5, 3.8), xytext=(5, 4.3),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Séparation P1 / hors P1 (les deux utilisés)
    ax2.add_patch(plt.Rectangle((0.5, 2.5), 4, 1), fc='#2ecc71', alpha=0.6, ec='black')
    ax2.text(2.5, 3, 'P1\n(vraies valeurs)', ha='center', va='center', fontsize=9)
    
    ax2.add_patch(plt.Rectangle((4.5, 2.5), 5, 1), fc='#f39c12', alpha=0.6, ec='black')
    ax2.text(7, 3, 'Hors P1\n(imputées)', ha='center', va='center', fontsize=9)
    
    # Flèches de fusion
    ax2.annotate('', xy=(5, 1.8), xytext=(2.5, 2.3),
                arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
    ax2.annotate('', xy=(5, 1.8), xytext=(7, 2.3),
                arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))
    
    # Entraînement
    ax2.add_patch(plt.Rectangle((2, 0.5), 6, 1), fc='#9b59b6', alpha=0.7, ec='black')
    ax2.text(5, 1, 'Modèle entraîné sur P1 + imputées', ha='center', va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    


print("=" * 80)
print("STRATÉGIES D'ENTRAÎNEMENT")
print("=" * 80)
visualize_training_strategies()

---

## 5 - Résumé et tableau récapitulatif

<a id="5---résumé-et-tableau-récapitulatif"></a>

### 5.1 - Tableau récapitulatif des paramètres

<a id="51---tableau-récapitulatif-des-paramètres"></a>

In [ ]:
# Tableau récapitulatif complet
params_summary = pd.DataFrame([
    {
        'Paramètre': 'target_frequency',
        'Type': 'str | Dict',
        'Défaut': '-',
        'Description': 'Fréquence cible pour l\'imputation (ex: "M", "Q")',
        'Impact': 'Détermine la granularité de sortie'
    },
    {
        'Paramètre': 'estimator',
        'Type': 'Estimator | Dict',
        'Défaut': '-',
        'Description': 'Modèle(s) pour prédire les valeurs manquantes',
        'Impact': 'Qualité des imputations'
    },
    {
        'Paramètre': 'keep_lower_frequencies',
        'Type': 'bool',
        'Défaut': 'True',
        'Description': 'Conserver les fréquences intermédiaires',
        'Impact': 'Structure de sortie (MultiIndex ou Index simple)'
    },
    {
        'Paramètre': 'cascade_refitting',
        'Type': 'bool',
        'Défaut': 'True',
        'Description': 'Réentraîner après chaque niveau de fréquence',
        'Impact': 'Précision vs Vitesse'
    },
    {
        'Paramètre': 'imputation_scope',
        'Type': 'Literal',
        'Défaut': 'strict',
        'Description': 'Extension de la fenêtre P1 pour l\'entraînement',
        'Impact': 'Taille du jeu d\'entraînement'
    },
    {
        'Paramètre': 'attrition_threshold',
        'Type': 'float [0-1]',
        'Défaut': '0.5',
        'Description': 'Seuil minimum de colonnes disponibles',
        'Impact': 'Étendue de la fenêtre d\'entraînement'
    },
    {
        'Paramètre': 'train_on_partial_coverage',
        'Type': 'bool',
        'Défaut': 'False',
        'Description': 'Utiliser les valeurs imputées pour l\'entraînement',
        'Impact': 'Qualité vs Quantité de données'
    },
    {
        'Paramètre': 'impute_delayed_values',
        'Type': 'bool',
        'Défaut': 'False',
        'Description': 'Imputer les valeurs affectées par des délais',
        'Impact': 'Gestion des données récentes'
    },
    {
        'Paramètre': 'delays',
        'Type': 'DataFrame | None',
        'Défaut': 'None',
        'Description': 'Tableau des délais de publication',
        'Impact': 'Précision du calcul de P1'
    }
])

print("=" * 100)
print("TABLEAU RÉCAPITULATIF DES PARAMÈTRES")
print("=" * 100)
display(params_summary.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap'
}))

### 5.2 - Recommandations par cas d'usage

<a id="52---recommandations-par-cas-dusage"></a>

In [ ]:
# Recommandations par cas d'usage
recommendations = pd.DataFrame([
    {
        'Cas d\'usage': '🚀 Prototypage rapide',
        'keep_lower_frequencies': False,
        'cascade_refitting': False,
        'imputation_scope': 'strict',
        'attrition_threshold': 0.5,
        'train_on_partial_coverage': False,
        'Justification': 'Configuration minimale, exécution rapide'
    },
    {
        'Cas d\'usage': '📊 Reporting multi-fréquence',
        'keep_lower_frequencies': True,
        'cascade_refitting': False,
        'imputation_scope': 'strict',
        'attrition_threshold': 0.7,
        'train_on_partial_coverage': False,
        'Justification': 'Conservation des niveaux pour analyse à plusieurs échelles'
    },
    {
        'Cas d\'usage': '🎯 Production haute précision',
        'keep_lower_frequencies': False,
        'cascade_refitting': True,
        'imputation_scope': 'extended_both',
        'attrition_threshold': 0.5,
        'train_on_partial_coverage': False,
        'Justification': 'Cascade complète avec extension de fenêtre'
    },
    {
        'Cas d\'usage': '🔬 Recherche économétrique',
        'keep_lower_frequencies': True,
        'cascade_refitting': True,
        'imputation_scope': 'extended_both',
        'attrition_threshold': 0.6,
        'train_on_partial_coverage': True,
        'Justification': 'Configuration maximale pour analyse complète'
    },
    {
        'Cas d\'usage': '📈 Données récentes avec délais',
        'keep_lower_frequencies': False,
        'cascade_refitting': True,
        'imputation_scope': 'extended_forward',
        'attrition_threshold': 0.4,
        'train_on_partial_coverage': True,
        'Justification': 'Extension vers le futur pour données récentes manquantes'
    },
    {
        'Cas d\'usage': '🗄️ Données historiques limitées',
        'keep_lower_frequencies': True,
        'cascade_refitting': True,
        'imputation_scope': 'extended_backward',
        'attrition_threshold': 0.3,
        'train_on_partial_coverage': True,
        'Justification': 'Seuil bas et extension vers le passé'
    }
])

print("=" * 120)
print("RECOMMANDATIONS PAR CAS D'USAGE")
print("=" * 120)
display(recommendations)

### 5.3 - Points clés à retenir

<a id="53---points-clés-à-retenir"></a>

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                    POINTS CLÉS À RETENIR                                               ║
╠═══════════════════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                                        ║
║  1. STRUCTURE DE SORTIE                                                                                ║
║     ─────────────────────                                                                              ║
║     • keep_lower_frequencies=False → Index simple, une seule fréquence                                 ║
║     • keep_lower_frequencies=True  → MultiIndex avec toutes les fréquences                             ║
║                                                                                                        ║
║  2. STRATÉGIE D'IMPUTATION                                                                             ║
║     ────────────────────────                                                                           ║
║     • cascade_refitting=False → Un seul entraînement, imputation directe                                  ║
║     • cascade_refitting=True  → Cascade avec réentraînement, plus précis                                  ║
║                                                                                                        ║
║  3. FENÊTRE D'ENTRAÎNEMENT                                                                             ║
║     ───────────────────────                                                                            ║
║     • P1 = Période où TOUTES les séries ont des vraies valeurs                                         ║
║     • imputation_scope étend P1 (before/after/both) selon attrition_threshold                          ║
║     • train_on_partial_coverage inclut ou non les valeurs imputées                                      ║
║                                                                                                        ║
║  4. TRAÇABILITÉ (PROVENANCE)                                                                           ║
║     ────────────────────────                                                                           ║
║     • ORIGINAL      : Valeurs originales non modifiées                                                 ║
║     • MODEL_ON_TRUE : Imputées par modèle entraîné sur vraies valeurs                                  ║
║     • MODEL_ON_MIXED: Imputées par modèle entraîné sur valeurs mixtes                                  ║
║     • AGGREGATED    : Obtenues par agrégation temporelle                                               ║
║                                                                                                        ║
║  5. BONNES PRATIQUES                                                                                   ║
║     ─────────────────                                                                                  ║
║     ✓ Commencer par une configuration simple (Scénario A)                                              ║
║     ✓ Augmenter progressivement la complexité si nécessaire                                            ║
║     ✓ Toujours vérifier la matrice de provenance                                                       ║
║     ✓ Valider les résultats sur un échantillon de test                                                 ║
║     ✓ Documenter les paramètres utilisés pour la reproductibilité                                      ║
║                                                                                                        ║
╚═══════════════════════════════════════════════════════════════════════════════════════════════════════╝
""")

# Arbre de décision visuel
print("\n")
print("=" * 100)
print("ARBRE DE DÉCISION POUR LE CHOIX DES PARAMÈTRES")
print("=" * 100)
print("""

                               ┌─────────────────────────────┐
                               │  Besoin de toutes les       │
                               │  fréquences en sortie ?     │
                               └─────────────┬───────────────┘
                                             │
                         ┌───────────────────┴───────────────────┐
                         │                                       │
                        OUI                                     NON
                         │                                       │
                         ▼                                       ▼
           ┌─────────────────────────┐           ┌─────────────────────────┐
           │ keep_lower_frequencies  │           │ keep_lower_frequencies  │
           │        = True           │           │        = False          │
           └───────────┬─────────────┘           └───────────┬─────────────┘
                       │                                     │
                       ▼                                     ▼
           ┌─────────────────────────┐           ┌─────────────────────────┐
           │ Besoin de précision     │           │ Besoin de précision     │
           │ maximale ?              │           │ maximale ?              │
           └───────────┬─────────────┘           └───────────┬─────────────┘
                       │                                     │
         ┌─────────────┴──────────────┐        ┌─────────────┴──────────────┐
         │                            │        │                            │
        OUI                          NON      OUI                          NON
         │                            │        │                            │
         ▼                            ▼        ▼                            ▼
  ┌─────────────┐            ┌─────────────┐  ┌─────────────┐       ┌─────────────┐
  │ SCÉNARIO D  │            │ SCÉNARIO B  │  │ SCÉNARIO C  │       │ SCÉNARIO A  │
  │ (complet)   │            │ (multi-freq)│  │ (précis)    │       │ (rapide)    │
  └─────────────┘            └─────────────┘  └─────────────┘       └─────────────┘

""")

---

## Conclusion

Ce notebook a présenté de manière détaillée le fonctionnement de la classe `HighFrequencyImputer` pour l'imputation de données à fréquences mixtes. Les points essentiels à retenir sont :

1. **La fenêtre P1** est centrale dans le processus : c'est la période où toutes les séries ont des vraies valeurs.

2. **Les paramètres clés** (`keep_lower_frequencies`, `cascade_refitting`, `imputation_scope`, `attrition_threshold`) permettent un contrôle fin du processus d'imputation.

3. **La matrice de provenance** permet de tracer l'origine de chaque valeur (originale, imputée par modèle sur vraies valeurs, imputée par modèle sur valeurs mixtes, ou agrégée).

4. **Le choix des paramètres** dépend du cas d'usage : prototypage rapide, production haute précision, ou recherche complète.

Pour aller plus loin, consultez la documentation complète du package et les exemples d'utilisation avancée.